# ML 입문 32 — 전처리 / 파이프라인 (Preprocessing & Pipeline)

> **대상**: 분류(30)·회귀(31)를 마친 단계.
> **목표**: 모델에 넣기 전에 데이터를 다듬는 **전처리**(스케일링·인코딩)와, 전처리+모델을 하나로 묶는 **`Pipeline`**(데이터 누수 방지) 익히기.

지금까지는 붓꽃처럼 **깨끗한** 데이터를 바로 모델에 넣었습니다. 하지만 실전 데이터(곧 만날 타이타닉)는 **스케일이 제각각이고, 글자(범주형)가 섞여 있고, 빈 값**이 있습니다. 모델이 잘 학습하려면 먼저 **전처리**가 필요해요. 20(numpy 표준화)·21(pandas)이 여기서 ML과 합쳐집니다.

## 0) 왜 전처리인가 — 스케일이 모델을 망친다

특성마다 **숫자 범위(스케일)가 다르면**, KNN·로지스틱 같은 "거리 기반" 모델이 큰 숫자 특성에 휘둘립니다.

예: wine 데이터는 특성 평균이 **0.36 ~ 746.89**로 천차만별 → 큰 특성이 거리 계산을 지배.

```python
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

w = load_wine()
X_train, X_test, y_train, y_test = train_test_split(
    w.data, w.target, test_size=0.3, random_state=42, stratify=w.target)

# 스케일링 없이
knn = KNeighborsClassifier().fit(X_train, y_train)
accuracy_score(y_test, knn.predict(X_test))    # ≈ 0.72  (낮음!)
```

> 같은 모델인데 스케일링만 하면 **0.72 → 0.94**로 뜁니다(아래). 전처리가 모델 성능을 좌우한다는 가장 직관적인 증거.

In [2]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

w = load_wine()
X_train, X_test, y_train, y_test = train_test_split(
    w.data, w.target, test_size=0.3, random_state=42, stratify=w.target
)

knn = KNeighborsClassifier().fit(X_train, y_train)
accuracy_score(y_test, knn.predict(X_test))

0.7222222222222222

## 1) 스케일링 — StandardScaler (20의 표준화!)

`StandardScaler`는 각 특성을 **평균 0, 표준편차 1**로 맞춥니다. **20에서 numpy로 손계산했던 `(x - mean) / std`가 바로 이것**입니다.

```python
from sklearn.preprocessing import StandardScaler
import numpy as np

X = np.array([[90], [80], [70], [60], [100]], dtype=float)

scaler = StandardScaler()
scaler.fit(X)              # 평균·표준편차를 "학습"
scaler.mean_              # [80.0]
scaler.scale_             # [14.14...]  (표준편차)

scaler.transform(X)[0]    # [0.7071]  ← (90-80)/14.14, 20에서 본 그 값!
```

> `fit`으로 평균·표준편차를 구하고 `transform`으로 변환. 둘을 한 번에 하는 `fit_transform`도 있습니다. 변환 후엔 모든 특성이 같은 스케일이라 거리 기반 모델이 공정하게 동작합니다.

> ⚠️ **결정적 규칙**: `fit`은 **train 데이터에만** 합니다. test의 평균·표준편차를 쓰면 "시험 정보를 미리 본" 셈(데이터 누수, leakage). 이 실수를 막아주는 게 다음의 `Pipeline`입니다.

In [3]:
from sklearn.preprocessing import StandardScaler
import numpy as np

X = np.array([[90], [80], [70], [60], [100]], dtype=float)

scaler = StandardScaler()
scaler

,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [4]:
scaler.fit(X)

StandardScaler()

In [5]:
scaler.mean_

array([80.])

In [6]:
scaler.scale_

array([14.14213562])

In [7]:
scaler.transform(X)[0]

array([0.70710678])

## 2) 범주형 인코딩 — OneHotEncoder

모델은 숫자만 이해합니다. `'개발'`, `'영업'` 같은 **글자(범주형)는 숫자로 바꿔야** 합니다.

```python
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

df = pd.DataFrame({"dept": ["개발", "영업", "개발", "인사"]})

enc = OneHotEncoder(sparse_output=False)
enc.fit_transform(df[["dept"]])
# [[1 0 0]    개발 → (개발=1, 영업=0, 인사=0)
#  [0 1 0]    영업
#  [1 0 0]    개발
#  [0 0 1]]   인사
enc.categories_[0]    # ['개발' '영업' '인사']
```

> **원-핫 인코딩**: 범주마다 열을 하나씩 만들어 해당하면 1, 아니면 0. "개발/영업/인사"를 3개 열로. (`개발=0, 영업=1, 인사=2`처럼 숫자만 매기면 모델이 "인사 > 영업 > 개발" 같은 크기 관계로 오해할 수 있어, 순서 없는 범주는 원-핫이 안전합니다.)

In [8]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

df = pd.DataFrame({"dept": ["개발", "영업", "개발", "인사"]})

df

,dept
0,개발
1,영업
2,개발
3,인사


In [11]:
enc = OneHotEncoder(sparse_output=False)
enc

,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a SciPy sparse matrix/arrayin ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide <encoder_infrequent_categories>`.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'error'
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide <encoder_infrequen

In [12]:
enc.fit_transform(df[["dept"]])

array([[1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.],
       [0., 0., 1.]])

In [13]:
enc.categories_[0]

array(['개발', '영업', '인사'], dtype=object)

## 3) Pipeline — 전처리 + 모델을 하나로 (핵심)

전처리와 모델을 **하나의 객체로 묶으면**, `fit`/`predict`를 한 번에 하고 **데이터 누수도 자동으로 막힙니다**.

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

pipe = Pipeline([
    ("scaler", StandardScaler()),       # 1단계: 스케일링
    ("knn", KNeighborsClassifier()),    # 2단계: 모델
])

pipe.fit(X_train, y_train)              # 스케일러 fit → 변환 → 모델 fit, 한 번에
pipe.predict(X_test)                    # test도 train 기준으로 변환 후 예측
accuracy_score(y_test, pipe.predict(X_test))   # ≈ 0.94  (0.72에서 상승!)
```

> Pipeline의 핵심 이점 두 가지:
> 1. **누수 방지**: `pipe.fit(X_train, ...)`은 스케일러를 **train으로만** fit하고, `predict` 때 test를 그 기준으로 변환. 직접 `scaler.fit(전체)`하는 실수를 원천 차단.
> 2. **간결함**: 전처리+모델이 **하나의 모델**처럼 동작 → `fit`/`predict`/`score` 그대로. 교차검증(33)·하이퍼파라미터 탐색에도 통째로 들어감.

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier()),
])
pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('knn', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value d

In [17]:
pipe.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...), ('knn', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](3,)","[0,1,2]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,13
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


In [18]:
pipe.predict(X_test)

array([0, 1, 0, 0, 0, 0, 2, 1, 1, 2, 2, 1, 2, 1, 0, 2, 1, 0, 2, 2, 1, 2,
       2, 2, 2, 2, 0, 1, 0, 2, 0, 1, 2, 1, 1, 2, 1, 1, 1, 0, 2, 0, 0, 0,
       0, 1, 1, 0, 2, 0, 1, 1, 2, 0])

In [19]:
accuracy_score(y_test, pipe.predict(X_test))

0.9444444444444444

## 대표 예제 — 스케일링 효과 비교

> wine 데이터에서 KNN을 (a) 스케일링 없이 (b) Pipeline으로 스케일링 후, 두 정확도를 비교하라.

```python
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

w = load_wine()
X_train, X_test, y_train, y_test = train_test_split(
    w.data, w.target, test_size=0.3, random_state=42, stratify=w.target)

# (a) 스케일링 없이
raw = KNeighborsClassifier().fit(X_train, y_train)
acc_raw = accuracy_score(y_test, raw.predict(X_test))

# (b) Pipeline으로 스케일링 + KNN
pipe = Pipeline([("sc", StandardScaler()), ("knn", KNeighborsClassifier())])
pipe.fit(X_train, y_train)
acc_pipe = accuracy_score(y_test, pipe.predict(X_test))

print(f"스케일링 없이: {acc_raw:.4f}")    # 0.7222
print(f"스케일링 후:   {acc_pipe:.4f}")    # 0.9444
```

> 가르칠 포인트: **같은 모델·같은 데이터인데 전처리 하나로 0.72 → 0.94.** 전처리는 "있으면 좋은" 게 아니라 **모델 성능의 핵심**입니다. 그리고 그걸 안전하게(누수 없이) 하는 표준 도구가 `Pipeline`. 타이타닉부터 모든 실전에서 이 패턴을 씁니다.

In [25]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

w = load_wine()


In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    w.data, w.target, test_size=0.3, random_state=42, stratify=w.target
)

In [27]:
raw = KNeighborsClassifier().fit(X_train, y_train)
raw

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float, default=2Power parameter for the Minkowski metric. When p = 1, this is equivalentto using manhattan_distance (l1), and euclidean_distance (l2) for p = 2.For arbitrary p, minkowski_distance (l_p) is used. This parameter is expectedto be positive.",2
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.Doesn't affect :meth:`fit` method.",None
Name,Type,Value
"classes_ classes_: array of shape (n_classes,)Class labels known to the classifier","ndarray[int64](3,)","[0,1,2]"
"effective_metric_ effective_metric_: str or callbleThe distance metric used. It will be same as the `metric` parameteror a synonym of it, e.g. 'euclidean' if the `metric` parameter set to'minkowski' and `p` parameter set to 2.",str,'eu...an'


In [29]:
acc_raw = accuracy_score(y_test, raw.predict(X_test))
acc_raw

0.7222222222222222

In [31]:
pipe = Pipeline([("sc", StandardScaler()), ("knn", KNeighborsClassifier())])
pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('sc', ...), ('knn', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",5
,"weights weights: {'uniform', 'distance'}, callable or None, default='uniform'Weight function used in prediction. Possible values:- 'uniform' : uniform weights. All points in each neighborhood are weighted equally.- 'distance' : weight points by the inverse of their distance. in this case, closer neighbors of a query point will have a greater influence than neighbors which are further away.- [callable] : a user-defined function which accepts an array of distances, and returns an array of the same shape containing the weights.Refer to the example entitled:ref:`sphx_glr_auto_examples_neighbors_plot_classification.py`showing the impact of the `weights` parameter on the decisionboundary.",'uniform'
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depen

In [32]:
pipe.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('sc', ...), ('knn', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](3,)","[0,1,2]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,13
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
Name,Type,Value


In [34]:
acc_pipe = accuracy_score(y_test, pipe.predict(X_test))
acc_pipe

0.9444444444444444

In [35]:
print(f"스케일링 없이: {acc_raw:.4f}")    # 0.7222
print(f"스케일링 후:   {acc_pipe:.4f}")    # 0.9444

스케일링 없이: 0.7222
스케일링 후:   0.9444


## 검증 (노트북에서 실행)

```python
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# StandardScaler == numpy 표준화
X = np.array([[90],[80],[70],[60],[100]], dtype=float)
sc = StandardScaler().fit(X)
assert abs(sc.mean_[0] - 80.0) < 1e-9
assert abs(sc.transform(X)[0,0] - 0.7071) < 1e-3

# 스케일링 효과 (wine)
w = load_wine()
Xtr, Xte, ytr, yte = train_test_split(
    w.data, w.target, test_size=0.3, random_state=42, stratify=w.target)
raw = KNeighborsClassifier().fit(Xtr, ytr)
pipe = Pipeline([("sc", StandardScaler()), ("knn", KNeighborsClassifier())]).fit(Xtr, ytr)
assert abs(accuracy_score(yte, raw.predict(Xte)) - 0.7222) < 0.001
assert abs(accuracy_score(yte, pipe.predict(Xte)) - 0.9444) < 0.001
print("통과 ✅")
```

In [36]:
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# StandardScaler == numpy 표준화
X = np.array([[90],[80],[70],[60],[100]], dtype=float)
sc = StandardScaler().fit(X)
assert abs(sc.mean_[0] - 80.0) < 1e-9
assert abs(sc.transform(X)[0,0] - 0.7071) < 1e-3

# 스케일링 효과 (wine)
w = load_wine()
Xtr, Xte, ytr, yte = train_test_split(
    w.data, w.target, test_size=0.3, random_state=42, stratify=w.target)
raw = KNeighborsClassifier().fit(Xtr, ytr)
pipe = Pipeline([("sc", StandardScaler()), ("knn", KNeighborsClassifier())]).fit(Xtr, ytr)
assert abs(accuracy_score(yte, raw.predict(Xte)) - 0.7222) < 0.001
assert abs(accuracy_score(yte, pipe.predict(Xte)) - 0.9444) < 0.001
print("통과 ✅")

통과 ✅


## 직접 풀어보기 (연습)

`TODO`를 채운 뒤 검증 셀을 실행하세요.

### 연습 1 — StandardScaler 변환값

> 배열을 `StandardScaler`로 변환해, 변환된 값들의 평균과 표준편차를 (round 6자리) 튜플로 반환하라. (표준화하면 평균≈0, 표준편차≈1)
> **힌트**: `fit_transform` 후 `.mean()`, `.std()`.

```python
import numpy as np
from sklearn.preprocessing import StandardScaler

def solution(values):
    X = np.array(values, dtype=float).reshape(-1, 1)
    # TODO: 변환 후 (round(mean,6), round(std,6)) 반환
    pass

assert solution([90, 80, 70, 60, 100]) == (0.0, 1.0)
print("통과 ✅")
```

### 연습 2 — 원-핫 인코딩 열 수

> 범주형 데이터를 원-핫 인코딩했을 때 생기는 열(범주) 개수를 반환하라.
> **힌트**: `OneHotEncoder().fit(...).categories_[0]`의 길이, 또는 변환 결과의 `.shape[1]`.

```python
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

def solution(categories):
    df = pd.DataFrame({"c": categories})
    # TODO: 원-핫 인코딩 후 열 개수 반환
    pass

assert solution(["개발", "영업", "개발", "인사"]) == 3       # 개발/영업/인사
assert solution(["A", "B", "A", "A"]) == 2                  # A/B
print("통과 ✅")
```

### 연습 3 — Pipeline 정확도 (도전)

> wine 데이터를 `random_state=42, test_size=0.3, stratify`로 분할하고, `StandardScaler + KNN` 파이프라인의 test 정확도를 (round 4자리) 반환하라.

```python
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

def solution():
    w = load_wine()
    # TODO: 분할 → Pipeline(스케일+KNN) 학습 → test 정확도
    pass

assert solution() == 0.9444
print("통과 ✅")
```